In [3]:
### This script is the complete workflow to get an explanation and diagnosis from eeg data

from preprocessing import preprocess_file, npz_to_raw, preprocess_raw
from biomarkers import extract_features_fast, extract_biomarkers, extract_biomarker_outputs, load_reference_ranges
from feature_extraction import get_mean_eegpt_features
from classification import get_diagnosis, diag2llm
from llm_reasoning import main
import time
import numpy as np

# setup
raw_eeg = "C:/Users/marks/Downloads/caueeg-dataset/caueeg-dataset/signal/edf/00145.edf"
age = 74

start = time.time()

# preprocess
eeg = preprocess_file(raw_eeg).get_data() 

###
condition = 'photic_stimulation' #"resting_eyes_closed", "resting_eyes_open", "auditory_task", "cognitive_task", "photic_stimulation", "unknown"

# Load the healthy/AD reference ranges for AD pattern predictions
reference = load_reference_ranges("biomarker_reference.npz")

# extract biomarkers + save report
biomarker_features, biomarker_names, biomarker_report = extract_biomarker_outputs(
    eeg,
    condition=condition,
    reference=reference,
    save_to="new_biomarkers.txt",
)

# classification
biomarkers = extract_features_fast(eeg, mode='whole_head')[0][0,:] # need list of numeric values or dict in form {'name_of_bm': val} 
prediction_from_biomarkers, confidence_from_biomarkers = get_diagnosis(biomarkers, age, mode='biomarkers')
features = get_mean_eegpt_features(eeg) # need  segmentsx64
prediction_from_embeddings, confidence_from_embeddings = get_diagnosis(features, age, mode='embeddings')
prediction_text = diag2llm(prediction_from_biomarkers, confidence_from_biomarkers, prediction_from_embeddings, confidence_from_embeddings)
with open ('prediction.txt', 'w') as f:
    f.write(prediction_text)

# explanation
main() 

end = time.time()
print(f"Elapsed time: {end - start:.4f} seconds")


Extracting EDF parameters from C:\Users\marks\Downloads\caueeg-dataset\caueeg-dataset\signal\edf\00145.edf...
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 183399  =      0.000 ...   916.995 secs...
Effective window size : 16.000 (s)
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 0.5 - 45 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 0.50
- Lower transition bandwidth: 0.50 Hz (-6 dB cutoff frequency: 0.25 Hz)
- Upper passband edge: 45.00 Hz
- Upper transition bandwidth: 11.25 Hz (-6 dB cutoff frequency: 50.62 Hz)
- Filter length: 845 samples (6.602 s)

EEG channel type selected for re-referencing
Applying average reference.
Applying a custom ('EEG',) reference.
Report saved to: new_biomarkers.txt
Here's a breakdown of t